## Pipeline for supervised modeling

In [ ]:
import os

# Check if it's in the correct directory
print("Current working directory:", os.getcwd())
path = os.path.abspath(os.path.join(os.getcwd(), '..', 'path.py'))
%run $path

##### Configure notebook

In [ ]:
# Import data
train_file = '../data/random/split_3/train.csv'
val_file = '../data/random/split_3/val.csv'
test_file = '../data/random/split_3/test.csv'

# Configure dataloader
batch_size=24

# Modeling parameters
architecture_type='mpnn'
use_uncertainty=False
lr = 0.001
n_trials=0

# Contrastive learning parameters
contrastive_kwargs = {
    "node_mask_ratio": 0.25,
    "edge_mask_ratio": 0.25,
    "min_fragment_size": 4,
}

contrastive_config = {
    "enabled": True,
    "beta_aug": 0.20,
    "alpha_global": 0.20,
    "alpha_local": 0.20,
    "global_temperature": 0.1,
    "local_temperature": 0.1,
    "tanimoto_lambda": 0.5,
    "projection_dim": 128,
    "global_hidden_dim": 256,
    "local_hidden_dim": 256,
    "fp_radius": 2,
    "fp_size": 2048,
}

# Save the trained model
model_directory = '../output/models'
params_directory = '../output/params'
filename = 'model'
fig_path1="../output/figures"

# Plot embeddings along the epochs
method="tSNE"
emb_path="../output/embeddings"
fig_path2="../output/figures"
fig_path3="../output/figures"


##### Load data

In [ ]:
from params import load_data

train_smiles, y_train = load_data(train_file)
val_smiles, y_val = load_data(val_file)
test_smiles, y_test = load_data(test_file)

print(f"Training data: {len(train_smiles)} samples")
print(f"Validation data: {len(val_smiles)} samples")
print(f"Test data: {len(test_smiles)} samples")

##### Building molecular graphs in data loaders

In [ ]:
from loaders import graph_loader, graph_info

train_loader, val_loader, test_loader = graph_loader(
    train_smiles, 
    val_smiles, 
    test_smiles, 
    y_train, 
    y_val, 
    y_test, 
    batch_size=batch_size,
    seed=42,
    contrastive=contrastive_config["enabled"],
    contrastive_eval=contrastive_config["enabled"],
    contrastive_kwargs=contrastive_kwargs)

node_dim, edge_dim, num_tasks = graph_info(train_loader)
print(f"Max number of atom features: {node_dim}")
print(f"Max number of bond features: {edge_dim}")
print(f"Number of tasks: {num_tasks}")

first_batch = next(iter(train_loader))
print("Contrastive batch keys:", list(first_batch.keys()))

##### Starting optuna optimization

In [ ]:
from optimizer import objective
from params import initialize_optuna

study = initialize_optuna()
study.optimize(lambda trial: objective(
    trial,
    node_dim,
    edge_dim, 
    train_loader, 
    val_loader, 
    num_tasks, 
    architecture_type=architecture_type, 
    use_uncertainty=use_uncertainty,
    lr=lr,
    contrastive_config=contrastive_config), 
    n_trials=n_trials)

best_params = study.best_params
best_trial = study.best_trial
print("Best hyperparameters:", best_params)

##### Run best model found in optuna optimization

In [ ]:
from utils import device
from optimizer import retrain
from save import save_model, save_params

best_params = study.best_params
model, best_val_loss, min_val_loss, train_losses, val_losses = retrain(
    best_params, 
    node_dim, 
    edge_dim, 
    train_loader, 
    val_loader, 
    num_tasks,
    architecture_type=architecture_type, 
    use_uncertainty=use_uncertainty,
    lr=lr,
    contrastive_config=contrastive_config)

save_model(model, model_directory, filename)
params_to_save = dict(best_params)
params_to_save["contrastive_config"] = contrastive_config
save_params(params_to_save, params_directory, filename, device)

##### Statistical analysis of best model

In [ ]:
from utils import device
from statistical import ModelEvaluator

ModelEvaluator(
    model,
    device,
    train_loader,
    val_loader,
    test_loader,
    calibration=False)

##### Visualize predicted vs experimental values

In [ ]:
from logits import visualize_logits

visualize_logits(
    model,
    train_loader,
    val_loader,
    test_loader,
    task_index=2,
    out_path=fig_path1)

##### Visualize embeddings along the epochs

In [ ]:
from embeddings import visualize_embeddings

visualize_embeddings(
    in_path=emb_path,
    epoch=92,
    method="tSNE",
    task_index=0,
    out_path=fig_path2)